In [1]:
import numpy as np, pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from noharm_fl.certify.bounds import clip_loss, eb_pvalue, betting_pvalue, fixed_sequence

df = pd.read_parquet("../data/interim/sparcs_2022_adults.parquet").reset_index(drop=True)
df["split"] = pd.read_parquet("../data/splits/split_seed0.parquet")["split"].values   # PILOT seed
df["dx_sys"] = df["ccsr_diagnosis_code"].str[:3]
cols = ["age_group", "gender", "race", "ethnicity", "type_of_admission",
        "emergency_department_indicator", "payment_typology_1", "dx_sys"]
X = OneHotEncoder(handle_unknown="ignore", min_frequency=50).fit_transform(df[cols].fillna("NA")).tocsr()
y = df["y"].values; h = df["permanent_facility_id"].values
tr = (df["split"] == "train").values; ce = (df["split"] == "cert").values
keep = df.groupby("permanent_facility_id")["y"].sum().loc[lambda s: s >= 100].index
fed = LogisticRegression(C=1.0, max_iter=300).fit(X[tr], y[tr])

B, alpha = 5.0, 0.05
blends = [0.25, 0.5, 0.75, 1.0]
eps_grid = [0.001, 0.005, 0.01, 0.02]
D = {}
for k in keep:
    mk = (h == k); a_, b_ = mk & tr, mk & ce
    loc = LogisticRegression(C=1.0, max_iter=300).fit(X[a_], y[a_])
    pl = loc.predict_proba(X[b_])[:, 1]; pf = fed.predict_proba(X[b_])[:, 1]
    l0 = clip_loss(pl, y[b_], B)
    D[k] = [clip_loss((1 - a) * pl + a * pf, y[b_], B) - l0 for a in blends]

rows = []
for eps in eps_grid:
    for name, fn in [("empirical-Bernstein", eb_pvalue), ("betting", betting_pvalue)]:
        sel = [fixed_sequence([fn(d, eps, -B, B, alpha) for d in D[k]], alpha) for k in keep]
        sel = np.array(sel)
        rows.append((eps, name, int((sel >= 0).sum()), len(keep), round(float(np.mean([blends[s] if s >= 0 else 0 for s in sel])), 3)))
print(pd.DataFrame(rows, columns=["eps", "bound", "hospitals_with_nonlocal", "of", "mean_blend"]))

     eps                bound  hospitals_with_nonlocal   of  mean_blend
0  0.001  empirical-Bernstein                        0  126       0.000
1  0.001              betting                        0  126       0.000
2  0.005  empirical-Bernstein                        0  126       0.000
3  0.005              betting                       17  126       0.083
4  0.010  empirical-Bernstein                        3  126       0.012
5  0.010              betting                       57  126       0.373
6  0.020  empirical-Bernstein                       31  126       0.183
7  0.020              betting                       99  126       0.750


In [1]:
import numpy as np, pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

res = pd.read_csv("../outputs/tables/harm_seed100_noapr.csv")

def harm_count(diff, se, eps):
    diff, se = np.asarray(diff, float), np.asarray(se, float)
    z = np.where(se > 0, (diff - eps) / np.where(se > 0, se, 1), -np.inf)
    p = 1 - stats.norm.cdf(z)
    return int(multipletests(p, alpha=0.05, method="holm")[0].sum())

rows = []
for eps in [0.005, 0.01, 0.02]:
    rows.append({
        "eps": eps,
        "certified_harmed(>eps)": harm_count(res[f"seldiff_eps{eps}"], res[f"selse_eps{eps}"], eps),
        "certified_worst_regret": res[f"seldiff_eps{eps}"].max(),
        "pooled_harmed(>eps)": harm_count(res["diff_b1.0"], res["se_b1.0"], eps),
        "pooled_worst_regret": res["diff_b1.0"].max(),
        "blend0.75_harmed(>eps)": harm_count(res["diff_b0.75"], res["se_b0.75"], eps),
    })
print(pd.DataFrame(rows).round(4))

     eps  certified_harmed(>eps)  certified_worst_regret  pooled_harmed(>eps)  \
0  0.005                       0                  0.0010                    1   
1  0.010                       0                  0.0047                    0   
2  0.020                       0                  0.0131                    0   

   pooled_worst_regret  blend0.75_harmed(>eps)  
0               0.0246                       1  
1               0.0246                       0  
2               0.0246                       0  


In [2]:
rows = []
for eps in [0.005, 0.01, 0.02]:
    rows.append({"eps": eps,
        "pooled_raw_share_above_eps": (res["diff_b1.0"] > eps).mean(),
        "blend0.75_raw_share_above_eps": (res["diff_b0.75"] > eps).mean(),
        "certified_raw_share_above_eps": (res[f"seldiff_eps{eps}"] > eps).mean(),
        "pooled_hospitals_above_eps": int((res["diff_b1.0"] > eps).sum()),
        "certified_hospitals_above_eps": int((res[f"seldiff_eps{eps}"] > eps).sum())})
print(pd.DataFrame(rows).round(4))
print(res.assign(big=res["diff_b1.0"] > 0.005).groupby("big")[["n_train", "deaths_train"]].median())

     eps  pooled_raw_share_above_eps  blend0.75_raw_share_above_eps  \
0  0.005                      0.2540                         0.0873   
1  0.010                      0.0952                         0.0317   
2  0.020                      0.0238                         0.0000   

   certified_raw_share_above_eps  pooled_hospitals_above_eps  \
0                            0.0                          32   
1                            0.0                          12   
2                            0.0                           3   

   certified_hospitals_above_eps  
0                              0  
1                              0  
2                              0  
       n_train  deaths_train
big                         
False   5058.0         169.5
True    5846.5         179.0
